In [1]:
import cv2
import numpy as np

cam = cv2.VideoCapture(1, cv2.CAP_DSHOW)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

duck_height_calib = 349
calib_dist = 30 #cm
# duck_height_real = 7 #cm


while (True):
    success, frame = cam.read(0)
    
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (15, 112, 92), (57, 255, 255))
    
    kernel = np.ones((7, 7), np.uint8)
    morph = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    connectivity = 4
    
    connectivity = 4  
    output = cv2.connectedComponentsWithStats(morph, connectivity, cv2.CV_32S)
    num_labels = output[0]
    labels = output[1]
    stats = output[2]
    filtered = np.zeros_like(mask)
    
    for i in range(1, num_labels):
        a = stats[i, cv2.CC_STAT_AREA]
        top = stats[i, cv2.CC_STAT_TOP]
        left = stats[i, cv2.CC_STAT_LEFT]
        width = stats[i, cv2.CC_STAT_WIDTH]
        height = stats[i, cv2.CC_STAT_HEIGHT]
        
        if (a >= 2000):
            filtered[np.where(labels == i)] = 255
            
            distance_exp = calib_dist / height * duck_height_calib
            
            cv2.putText(frame, str(round(distance_exp, 1)) + " cm", (100, 100), cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 0, 255), 2, cv2.LINE_4)
            cv2.putText(frame, str(height), (left, top), cv2.FONT_ITALIC, 1, (0, 0, 255), 2, cv2.LINE_4)
            cv2.rectangle(frame, (left, top), (left + width, top + height), (0, 255, 0), 1)   
            
    
    # mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    # morph_bgr = cv2.cvtColor(morph, cv2.COLOR_GRAY2BGR)
    # filtered_bgr = cv2.cvtColor(filtered, cv2.COLOR_GRAY2BGR)
    
    # frames = [frame, mask_bgr, morph_bgr, filtered_bgr]
    # combined_image = np.hstack(frames)
    
    # cv2.namedWindow('custom fr_1', cv2.WINDOW_KEEPRATIO)
    # cv2.imshow('custom fr_1', combined_image)
    # cv2.resizeWindow('custom fr_1', 1080, 1080)
    
    cv2.namedWindow('custom fr_1', cv2.WINDOW_KEEPRATIO)
    cv2.imshow('frame', frame)
    cv2.resizeWindow('custom fr_1', 1080, 1080)
    
    
    key = cv2.waitKey(80) & 0xFF
    if (key == ord('q')):
        break

cam.release()
cv2.destroyAllWindows()
cv2.waitKey(10)

-1